# 02 - Preprocessing and augmentation

Create the stratified 70/15/15 split (duplicates removed to avoid leakage) and preview the augmentation applied to the **training set only**.

**Running on Google Colab?** Upload or clone the project, then run once:

```python
from google.colab import drive
drive.mount("/content/drive")          # optional: keep data and models on Drive
%cd /content/iClaim
%pip install -q -r requirements.txt
```
Use a GPU runtime (Runtime > Change runtime type) for training.

In [ ]:
# --- Setup (works locally and in Google Colab) ---
import os, sys
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

import warnings
warnings.filterwarnings("ignore")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
sns.set_theme(style="whitegrid")

from src.config import load_config, resolve_path, set_seed
cfg = load_config()
set_seed(cfg["project"]["seed"])
print("Project root:", ROOT)

## Create splits

In [ ]:
from src.data_loader import prepare_splits

manifest = prepare_splits(cfg, overwrite=True)   # raises a helpful error if data/raw is empty
manifest.groupby(["split", "class"]).size().unstack(fill_value=0)

## Preview augmentation

In [ ]:
import tensorflow as tf
from PIL import Image
from src.data_loader import build_augmentation

aug = build_augmentation(cfg)
sample = manifest[manifest["split"] == "train"].sample(1, random_state=cfg["project"]["seed"]).iloc[0]
h, w = cfg["data"]["image_size"]
img = np.asarray(Image.open(sample["file"]).convert("RGB").resize((w, h)), dtype="float32")

fig, axes = plt.subplots(2, 4, figsize=(12, 6))
axes[0][0].imshow(img.astype("uint8")); axes[0][0].set_title("Original")
for ax in axes.ravel()[1:]:
    ax.imshow(np.clip(aug(img[None], training=True)[0].numpy(), 0, 255).astype("uint8"))
for ax in axes.ravel():
    ax.axis("off")
plt.suptitle(f"Augmentation examples ({sample['class']})")
plt.tight_layout()
plt.savefig(resolve_path(cfg["paths"]["figures_dir"]) / "augmentation_examples.png", dpi=120)
plt.show()

## Class weights

Used during training when `training.use_class_weights` is true.

In [ ]:
from src.data_loader import compute_class_weights
weights = compute_class_weights(cfg)
pd.DataFrame({"class": cfg["data"]["class_names"], "weight": [round(weights[i], 3) for i in range(len(weights))]})